## Modal analysis with modekit

### Example 1 - Tap tests on the GARTEUR benchmark: experimental modal analysis, then an output-only analysis

This notebook demonstrates the modekit workflow on hammer tests of the aluminium GARTEUR structure, recorded in three states: healthy, damaged (bolts loosened on the right-hand side) and restored. In the first part we proceed as in an experimental modal analysis (EMA): we inspect the hammer and response signals and discard the faulty repetitions, apply a force window and an exponential window, estimate the frequency response functions and the coherence, and identify the modes band by band with pLSCF followed by an LSFD residue fit. In the second part we take records acquired with a fan blowing on the structure and run the same identification on output spectra alone, as one would with ambient vibration data.

The measurements come from the [modekit-data](https://github.com/garavanis/modekit-data) repository. On its first run the notebook downloads the sets it needs (about 450 MB) into `data/` next to it; the tables and the cache of the fits go to `outputs/garteur/`, the figures to `figures/garteur/`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

from modekit import taptest

First, we load the tap-test sets. `taptest.DataLoader` reads every CSV of the folder into a `TapTest`, which holds the samples of a set as an array of shape `(n_reps, n_channels, n_samples)` together with the channel names and the sample rate. One file is one set: repeated taps at one point of the structure, in one state.

In [ ]:
from pathlib import Path
from modekit import datasets

# the test sets this notebook uses, downloaded on the first run from the
# modekit-data repository into data/ next to it
TAP_SETS = [
    "fr_tap_hs1", "fr_tap_hs2", "fr_tap_hsr", "fr_tap_ds",
    "rw_tap_hs1", "rw_tap_hs2", "rw_tap_hsr", "rw_tap_ds",
    "fanfr_nb_hs", "fanfr_nb_ds", "fanfr_b0_hs", "fanfr_b0_ds", "fanfr_b45_hs", "fanfr_b45_ds",
]
HERE = Path.cwd()   # the examples folder: Jupyter runs a notebook in its own folder
DATA_DIR = datasets.fetch("heartspace", [f"{s}.csv" for s in TAP_SETS], HERE / "data")
OUT_DIR = HERE / "outputs" / "garteur"   # tables and the MPE cache
FIG_DIR = HERE / "figures" / "garteur"
data_loader = taptest.DataLoader(DATA_DIR)
data = data_loader.load_data()

## Experimental modal analysis of the hammer tests

We use eight sets. Their names give the impact point (`fr` front, `rw` right wing), the state of the structure (`hs` healthy, `ds` damaged, `hsr` restored) and the batch (`hs1`, `hs2`). Every set holds four to six repetitions of the hammer force and the responses of four accelerometers: nose, tail, starboard wing and port-side wing. We gather the sets in a dictionary and keep this order throughout.

In [ ]:
# save in dictionary for use throughout
taps = {
    "fr_hs1": data["fr_tap_hs1"],
    "fr_hs2": data["fr_tap_hs2"],
    "fr_hsr": data["fr_tap_hsr"],
    "rw_hs1": data["rw_tap_hs1"],
    "rw_hs2": data["rw_tap_hs2"],
    "rw_hsr": data["rw_tap_hsr"],
    "fr_ds":  data["fr_tap_ds"],
    "rw_ds":  data["rw_tap_ds"],
}

# the response channels, in data order (Front_255, Tail_264, right_wing_484, left_wing_258)
SENSORS = ["nose", "tail", "starboard", "port-side"]

A set is indexed by channel name: `load_channel` returns the `(n_reps, n_samples)` array of one channel, and `data` the full block.

In [ ]:
taps["fr_hs1"].load_channel("Hammer").shape

In [ ]:
taps["fr_hs1"].data.shape

We start with the hammer. The figure below shows the force of every repetition of the *fr_hs1* set (the subscript of $f$ is the repetition). The fourth repetition contains a second, smaller impact shortly after the main one: a double hit. We will discard it, because a double hit distorts the input spectrum, as the *Input power spectra* section shows.

In [ ]:
from modekit.plots import plot_signal

plt.style.use("modekit.mathtext")  # Computer Modern maths, no LaTeX needed; modekit.latex with a LaTeX installation

fs = taps["fr_hs1"].sample_rate
print(f"Sample rate: {fs} Hz")

# Time vector from the sample rate (shared across all tap sets since we trimmed to the shortest rep)
nt = taps["fr_hs1"].data.shape[2]
time_array = np.arange(nt) / fs

hammer_hs1 = taps["fr_hs1"].load_channel("Hammer")   # (reps, nt)
rep_labels = [rf"$f_{{{r}}}(t)$ [N]" for r in range(1, hammer_hs1.shape[0] + 1)]
fig, axes = plot_signal(
    time_array,
    hammer_hs1,
    x_label="Time [s]",
    y_labels=rep_labels,
    figsize=(6, 10),
    save_fig_name=FIG_DIR / "ema/hammer_fr_hs1.pdf",
)

The impulse lasts well under a millisecond, so we zoom in on the time axis to see the shape of the pulse.

In [ ]:
fig, axes = plot_signal(time_array, hammer_hs1, x_label="Time [s]",
    y_labels=rep_labels, xlim=(0.1, 0.5), figsize=(4, 8),
    save_fig_name=FIG_DIR / "ema/hammer_fr_hs1_pulse.pdf")

Zooming in further, between the taps, shows the noise level of the force channel.

In [ ]:
_, axes = plot_signal(time_array, hammer_hs1, x_label="Time [s]",
    y_labels=rep_labels, xlim=(1.5, 1.525), ylim=(-0.15, 0.15), figsize=(4, 8),
    save_fig_name=FIG_DIR / "ema/hammer_fr_hs1_noise.pdf")

The other sets can be inspected the same way, and none of them shows a double impact. The cell is commented out to keep the notebook short; uncomment it to see the plots.

In [ ]:
# # Uncomment  to plot the tap test sets

# figs = {}
# for name, tap in taps.items():
#     fig, axes = plot_signal(
#         time_array,
#         tap.load_channel("Hammer"),
#         x_label="Time [s]",
#         y_labels=[rf"$f_{{{r}}}(t)$ [N]" for r in range(1, len(tap.load_channel("Hammer")) + 1)],
#         figsize=(6, 10),
#     )
#     fig.suptitle(name)
#     figs[name] = (fig, axes)

# # Per-plot axis tweaks 
# figs["rw_hs1"][1][0].set_ylim(-1, 1); figs["rw_hs1"][1][0].set_xlim(0.9, 1)
# figs["rw_hs2"][1][1].set_ylim(-2, 1)

Visual inspection does not scale, so we also run `taptest.find_bad_reps` on every set. On the hammer channel (`signal='i'`) it looks for clipping and for a second impact within the pulse; on the response channels (`signal='o'`) it looks for clipping only. A repetition flagged by either check is marked bad on its set and left out of everything that follows. A flag is worth confirming on the time signals, which is what we did above.

In [ ]:
# flag bad reps on each tap set
for name, tap in taps.items():
    bad_f = taptest.find_bad_reps(x=tap.load_channel("Hammer"), fs=fs, signal='i', print_summary=True, name=name)
    bad_r = taptest.find_bad_reps(x=tap.data[:, 1:, :],         fs=fs, signal='o', print_summary=True, name=name)
    tap.bad_reps = bad_f.any(axis=1) | bad_r.any(axis=1)

if not any(tap.bad_reps.any() for tap in taps.values()):
    print("All sets clean")

### Input power spectra

We compute the power spectrum of the force of each repetition with the FFT and average the spectra of a set to reduce the random error. We compute a spectrum rather than a spectral density: a density assumes a stationary signal, whereas a hammer impact is a transient, and spreading its finite energy over the record length would make the result depend on how long we recorded. The power spectrum keeps the energy the impact delivered, whatever the record length.

`EmaModel` holds the force and the responses of a set and computes its spectra, FRFs and coherence. `exc_type="transient"` selects this treatment of the input: one FFT per repetition, no segmenting of the record, the average taken across the repetitions.

#### Front taps

In [ ]:
from modekit.freq_models import EmaModel
import equinox as eqx

tap_models = {
    name: EmaModel(
        X=taps[name].load_channel("Hammer")[:, None, :],
        Y=taps[name].data[:, 1:, :],
        fs=fs,
        exc_type="transient",
    )
    for name in taps
}

print("\nEmaModel fields:")
print("-" * 30)
eqx.tree_pprint(tap_models["fr_hs1"])

In [ ]:
from collections import namedtuple

# called psd for convenience but here it is actually power spectra
PSD = namedtuple("PSD", "freqs Sxx Syy")
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

We plot the averaged input power spectra of the three front sets, over the full band and zoomed on 0–800 Hz. As expected, the spectra roll off with frequency: the soft hammer tip concentrates the energy at low frequencies, and above roughly 2 kHz the force spectrum is noise. In the zoomed view the *fr_hs1* set shows a comb-filter pattern, regularly spaced notches: the signature of the double hit in its fourth repetition.

In [ ]:
fig,axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{f}$ [N$^2$]"],
    yscale="log",
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front.pdf",
)

# zoomed-view
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{f}$ [N$^2$]"],
    yscale="log",
    xlim=(0, 800),
    ylim=(1e-6, 2e-2),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front_zoom.pdf",
)
plt.show()

We rebuild the *fr_hs1* model from the good repetitions only and plot the zoomed view again: the notches are gone.

In [ ]:
# rebuild tap_models["fr_hs1"]
tap_models["fr_hs1"] = EmaModel(X=taps["fr_hs1"].load_channel("Hammer", good=True)[:, None, :], 
                                Y=taps["fr_hs1"].good[:, 1:, :], fs=fs, exc_type="transient")
# rebuild psd["fr_hs1"] from the updated model
psd["fr_hs1"] = PSD(*tap_models["fr_hs1"].get_psd())

In [ ]:
# Plot again 
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{f}$ [N$^2$]"],
    yscale="log",
    xlim=(0, 800),
    ylim=(1e-6, 2e-2),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front_zoom_good.pdf",
)
plt.show()

#### Right-wing taps

The right-wing sets go the same way. We print one model to see its fields and plot the three input spectra.

In [ ]:
eqx.tree_pprint(tap_models["rw_hsr"])

In [ ]:
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Sxx,
        "R-wing tap HS2": psd["rw_hs2"].Sxx,
        "R-wing tap HSR": psd["rw_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{f}$ [N$^2$]"],
    yscale="log",
    # xlim=(0, 800),
    # ylim=(1e-7, 2e-2),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_rwing.pdf",
)
plt.show()

### Force window

The force is zero before and after the pulse, so whatever the input channel records there is noise. A force window keeps the pulse and suppresses the rest: a rectangle around the impact with cosine tapers at both ends (`windows.Tukey`), the form Avitabile (2018, p. 137) recommends. We rebuild every model with the windowed force, dropping the bad repetitions of each set, and plot the input spectra once more.

In [ ]:
from modekit import windows

def _fw(x, window=None, enabled=True):
    if window is None:
        window = windows.Tukey(pre_ms=5.0, post_ms=5.0, taper_ms=2.0)
        
    return window.apply(x, fs=fs) if enabled else x

# rebuild every model with the (windowed) input, dropping bad reps per set
tap_models = {
    name: EmaModel(
        X=_fw(tap.load_channel("Hammer", good=True))[:, None, :],
        Y=tap.good[:, 1:, :],
        fs=fs,
        exc_type="transient",
    )
    for name, tap in taps.items()
}
# rebuild the power spectra from the updated models
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

In [ ]:
# FRONT
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{f}$ [N$^2$]"],
    yscale="log",
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front_force_win.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Sxx,
        "R-wing tap HS2": psd["rw_hs2"].Sxx,
        "R-wing tap HSR": psd["rw_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{f}$ [N$^2$]"],
    yscale="log",
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_rwing_force_win.pdf",
)
plt.show()

### Response signals

We turn to the responses. The grid below shows every repetition of the *fr_hs1* set, one column per accelerometer.

In [ ]:
cmap = plt.get_cmap("Dark2")

figs = {}
# for name, tap in taps.items():
for name, tap in {k: taps[k] for k in ["fr_hs1"]}.items():
    n_reps = tap.data.shape[0]
    n_cols = len(tap.channels[1:])

    fig, axes = plt.subplots(
        n_reps, n_cols,
        figsize=(4 * n_cols, 2 * n_reps),
        sharex=True, squeeze=False,
    )
    for j, ch in enumerate(tap.channels[1:]):
        sig = tap.load_channel(ch)
        color = cmap(j)   # one color per sensor 
        for i in range(n_reps):
            axes[i, j].plot(time_array, sig[i, :], color=color, linewidth=1.0)
            axes[i, j].grid(True, linestyle="--", alpha=0.7)
        axes[0, j].set_title(ch)

    for i in range(n_reps):
        axes[i, 0].set_ylabel(f"rep {i + 1}")
    for j in range(n_cols):
        axes[-1, j].set_xlabel("Time [s]")

    fig.suptitle(name)
    fig.tight_layout()
    save_path = FIG_DIR / f"ema/responses_{name}.pdf"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    figs[name] = (fig, axes)

The response power spectra of one sensor across the sets (`sid` selects the sensor; here the port-side wing):

In [ ]:
# 0 -> "Front_255"
# 1 -> "Tail_264"
# 2 -> "right_wing_484"
# 3 -> "left_wing_258"
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Syy[sid:sid + 1],
        "Front tap HS2": psd["fr_hs2"].Syy[sid:sid + 1],
        "Front tap HSR": psd["fr_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{sid + 1}}}}}$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 800),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Syy[sid:sid + 1],
        "R-wing tap HS2": psd["rw_hs2"].Syy[sid:sid + 1],
        "R-wing tap HSR": psd["rw_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{sid + 1}}}}}$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 800),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

In the right-wing spectra one curve stands out: repetition 5 of *rw_hs1* has a glitch on the port-side sensor that `find_bad_reps` does not catch, since it looks for clipping and double hits only. We flag that repetition by hand with `flag(4)` (0-based) and rebuild the models.

In [ ]:
# find_bad_reps doesn't catch this kind of glitch,
# flag rep 5 of rw_hs1 manually before rebuilding
taps["rw_hs1"].flag(4)

# rebuild again dropping rep 5 of rw_tap_hs1
tap_models = {
    name: EmaModel(
        X=_fw(tap.load_channel("Hammer", good=True))[:, None, :],
        Y=tap.good[:, 1:, :],
        fs=fs,
        exc_type="transient",
    )
    for name, tap in taps.items()
}
# rebuild the power spectra from the updated models
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

In [ ]:
# Plot again

# 0 -> "Front_255"
# 1 -> "Tail_264"
# 2 -> "right_wing_484"
# 3 -> "left_wing_258"
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Syy[sid:sid + 1],
        "Front tap HS2": psd["fr_hs2"].Syy[sid:sid + 1],
        "Front tap HSR": psd["fr_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{sid + 1}}}}}$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_front_{SENSORS[sid]}_good.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Syy[sid:sid + 1],
        "R-wing tap HS2": psd["rw_hs2"].Syy[sid:sid + 1],
        "R-wing tap HSR": psd["rw_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{sid + 1}}}}}$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_rwing_{SENSORS[sid]}_good.pdf",
)
plt.show()

### Exponential window on the responses

A lightly damped structure keeps ringing after the record ends, which leaks energy across the spectrum. The exponential window stops the leakage, at a price: it adds damping, which has to be taken out of the identified poles afterwards. modekit does that through the `alpha` argument of the estimators.

We therefore do not apply it blindly. We first check whether the responses decay to zero within the record, look at the FRFs and the coherence for the symptoms of leakage (smeared peaks, sharp coherence drops at the resonances), and window only if there is leakage.

Before applying it we have to choose the decay rate $\alpha$. The rule is to add as little damping as possible (Avitabile 2018, p. 150). A practical way is to size $\alpha$ for the worst case: find the response that decays the least and choose $\alpha$ so that it reaches a target level by the end of the record.

`windows.Exponential.from_tap_tests` does exactly that for a set, from its good repetitions; here the target is 1 % of the peak (−40 dB) at the end of the record. Each set gets its own window, since the sets are independent tests. The pre-trigger can also be zeroed. That removes numerical noise from an otherwise clean start of the record and enforces the zero initial condition the transfer function $H(s) = X(s)/F(s)$ assumes (p. 281), but it does not remove the ring-down of a previous tap; the only cure for that is enough dwell time between taps.

We plot the responses of one set before and after the window.

In [ ]:
# Each tap set is an independent SIMO, so size its own exponential window.
# tap.good already drops the bad reps, so alpha is sized on clean reps only.
resp_win = {
    name: windows.Exponential.from_tap_tests(
        tap.good,
        fs,
        target_x=0.01,
        pre_ms=1.0,
        tail_frac=0.05,
        smooth_ms=20.0,
        zero_pretrigger=False,
    )
    for name, tap in taps.items()
}


def _rw(x, ref, name, windows=resp_win, enabled=True):
    """Apply the exponential window sized for tap set `name`."""
    return windows[name].apply(x, fs=fs, ref=ref) if enabled else x


for name, win in resp_win.items():
    print(f"{name}: response window sized (alpha={win.alpha:.4f} 1/s)")

In [ ]:
cmap = plt.get_cmap("Dark2")

figs = {}
# for name, tap in taps.items():
for name, tap in {k: taps[k] for k in ["rw_ds"]}.items():
    n_reps = tap.data.shape[0]
    n_cols = len(tap.channels[1:])
    hammer = tap.load_channel("Hammer")  # reference for the exponential window

    fig, axes = plt.subplots(
        n_reps, n_cols,
        figsize=(4 * n_cols, 2 * n_reps),
        sharex=True, squeeze=False,
    )
    for j, ch in enumerate(tap.channels[1:]):
        sig = tap.load_channel(ch)
        sig_win = _rw(sig, ref=hammer, name=name)  # exponentially decayed response
        color = cmap(j)   # one color per sensor
        for i in range(n_reps):
            axes[i, j].plot(time_array, sig[i, :], color=color, linewidth=1.0,
                            alpha=0.4, label="Original")
            axes[i, j].plot(time_array, sig_win[i, :], color=color, linewidth=1.0,
                            label="Windowed")
            axes[i, j].grid(True, linestyle="--", alpha=0.7)
        axes[0, j].set_title(ch)

    for i in range(n_reps):
        axes[i, 0].set_ylabel(f"rep {i + 1}")
    for j in range(n_cols):
        axes[-1, j].set_xlabel("Time [s]")

    axes[0, 0].legend(loc="upper right", fontsize="small")
    fig.suptitle(name)
    fig.tight_layout()
    save_path = FIG_DIR / f"ema/responses_{name}_exp_win.pdf"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    figs[name] = (fig, axes)

With both windows in place we rebuild every model: the force window on the input, the exponential window on each response, referenced to the hammer so that the decay starts at the impact. We plot the response spectra again, for a sensor on the starboard wing this time and including the damaged sets.

In [ ]:
# build the new EmaModel here: force window on the input, exponential
# window on the responses
tap_models = {}
for name, tap in taps.items():
    hammer = tap.load_channel("Hammer", good=True)
    Y_raw = tap.good[:, 1:, :]
    # _rw works per (n_reps, n_samples) channel, so window each and stack back
    Y_win = np.stack(
        [_rw(Y_raw[:, c, :], ref=hammer, name=name, enabled=True) for c in range(Y_raw.shape[1])],
        axis=1, 
    )
    tap_models[name] = EmaModel(
        X=_fw(hammer)[:, None, :],
        Y=Y_win,
        fs=fs,
        exc_type="transient",
    )

# rebuild the power spectra from the windowed models
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

In [ ]:
# Plot again

# 0 -> "Front_255"
# 1 -> "Tail_264"
# 2 -> "right_wing_484"
# 3 -> "left_wing_258"
sid = 2 

# FRONT TAP
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Syy[sid:sid + 1],
        "Front tap HS2": psd["fr_hs2"].Syy[sid:sid + 1],
        "Front tap HSR": psd["fr_hsr"].Syy[sid:sid + 1],
        "Front tap DS": psd["fr_ds"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{sid + 1}}}}}$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 250),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_front_{SENSORS[sid]}_exp_win.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Syy[sid:sid + 1],
        "R-wing tap HS2": psd["rw_hs2"].Syy[sid:sid + 1],
        "R-wing tap HSR": psd["rw_hsr"].Syy[sid:sid + 1],
        "R-wing tap DS": psd["rw_ds"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{sid + 1}}}}}$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 250),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_rwing_{SENSORS[sid]}_exp_win.pdf",
)
plt.show()

### Frequency response functions

Each set is a single-input, multiple-output test ($P = 1$, $Q = 4$), so we estimate its FRFs with the H1 estimator on its own model. We plot the magnitude of one sensor across the sets, then, in the low band, the real part, the imaginary part and the phase: the real part crosses zero and the imaginary part peaks at each natural frequency.

In [ ]:
# Each tap set is an independent SIMO test (P=1, Q=4), so we compute its FRF on its
# own model. 
FRF = namedtuple("FRF", "freqs H")
frf = {name: FRF(*m.get_frf(method="H1")) for name, m in tap_models.items()}

In [ ]:
# Plot FRF magnitude 
sid = 2
# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        "Front tap HS1": np.abs(frf["fr_hs1"].H[sid]),
        "Front tap HS2": np.abs(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.abs(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.abs(frf["fr_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$|H_{{{sid + 1}}}|$ [(m/s$^2$)/N]"],
    yscale="log",
    xlim=(0, 800),
    # ylim=(5e-2, 1e1),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_mag_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.abs(frf["rw_hs1"].H[sid]),
        "R-wing tap HS2": np.abs(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.abs(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.abs(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$|H_{{{sid + 1}}}|$ [(m/s$^2$)/N]"],
    yscale="log",
    xlim=(0, 800),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_mag_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

In [ ]:
# Plot the FRF real part: it crosses through zero at the natural frequency
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        "Front tap HS1": np.real(frf["fr_hs1"].H[sid]),
        "Front tap HS2": np.real(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.real(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.real(frf["fr_ds"].H[sid])
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\Re\,H_{{{sid + 1}}}$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(-2.5, 1.5),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_real_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.real(frf["rw_hs1"].H[sid]),
        "R-wing tap HS2": np.real(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.real(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.real(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\Re\,H_{{{sid + 1}}}$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(-200,200), 
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_real_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

In [ ]:
# Plot the FRF imaginary part: it reaches its peak at the natural frequency
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        "Front tap HS1": np.imag(frf["fr_hs1"].H[sid]),
        "Front tap HS2": np.imag(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.imag(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.imag(frf["fr_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\Im\,H_{{{sid + 1}}}$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    ylim=(-3, 3),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_imag_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.imag(frf["rw_hs1"].H[sid]),
        "R-wing tap HS2": np.imag(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.imag(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.imag(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\Im\,H_{{{sid + 1}}}$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_imag_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

In [ ]:
# Plot the FRF phase
sid = 2

# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "Front tap HS1": np.angle(frf["fr_hs1"].H[sid]),
        # "Front tap HS2": np.angle(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.angle(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.angle(frf["fr_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\angle H_{{{sid + 1}}}$ [rad]"],
    yscale="linear",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_phase_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.angle(frf["rw_hs1"].H[sid]),
        # "R-wing tap HS2": np.angle(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.angle(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.angle(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\angle H_{{{sid + 1}}}$ [rad]"],
    yscale="linear",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_phase_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

### Coherence

The ordinary coherence between the force and a response says how much of that response is linearly explained by the input at each frequency. It is close to one where the measurement is good and drops at the anti-resonances and where the hammer delivers little energy.

In [ ]:
COH = namedtuple("COH", "freqs gamma")
coh = {name: COH(*m.get_coherence(method="ordinary")) for name, m in tap_models.items()}

In [ ]:
# Plot the coherence
sid = 3

# FRONT TAP
fig, axes = plot_signal(
    coh["fr_hs1"].freqs,
    {
        "Front tap HS1": coh["fr_hs1"].gamma[sid],
        "Front tap HS2": coh["fr_hs2"].gamma[sid],
        "Front tap HSR": coh["fr_hsr"].gamma[sid],
        "Front tap DS": coh["fr_ds"].gamma[sid]
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\gamma^2_{{{sid + 1}}}$ [-]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(0, 1.05),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/coherence_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    coh["fr_hs1"].freqs,
    {
        "R-wing tap HS1": coh["rw_hs1"].gamma[sid],
        "R-wing tap HS2": coh["rw_hs2"].gamma[sid],
        "R-wing tap HSR": coh["rw_hsr"].gamma[sid],
        "R-wing tap DS": coh["rw_ds"].gamma[sid]
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$\gamma^2_{{{sid + 1}}}$ [-]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(0, 1.05),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/coherence_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

## Modal parameter estimation

When estimating modal parameters it pays to work in small frequency bands, each holding only a few modes.

Poly-reference Least-Squares Complex Frequency (pLSCF, "PolyMAX") fits a discrete-time model. Its basis is $z^k$ with $z = e^{j\omega\,dt}$ and $dt = 1/f_s$. At $f_s = 16384$ Hz the time step is so small that even the highest frequency of our band of interest (up to about 800 Hz) changes $z$ only slightly: the whole band occupies a tiny arc of the unit circle near $z = 1$, the basis vectors $z^0, z^1, \ldots, z^p$ crowd into the same neighbourhood, and the fit cannot tell them apart, so it finds no poles in the band. Cropping the frequency axis to the band does not help, because $dt$ is unchanged.

The remedy is a fictitious sampling rate $f_s^{\mathrm{fake}}$ just above the upper edge of each sub-band, for example 300 Hz for a 2–120 Hz band. The fictitious time step is much larger, the same band maps to a wide arc of the unit circle, the basis spreads out and the fit is well conditioned. The poles found on this stretched axis are converted back to physical frequencies and damping with the true $dt$, through $s = \ln(z)/dt$, so the identified values are correct.

This is the subdomain curve fitting of Avitabile (Ch. 5 and 9): fit small bands with a few modes each, then stitch the results together.

In [ ]:
from modekit.bandmpe import full_band_mpe
from modekit import reporting as rp
from modekit.export import save_table

# --- MPE settings (part of the cache identity: change any -> refit) ---
# Soft criteria / Hard criteria
SC = {"err_fn": 0.01, "err_zeta": 0.05, "err_phi": 0.05}
HC = {"zeta_max": 0.1, "mpc_lim": 0.5, "mpd_lim": 0.5}

# (f_lo, f_hi, fs_fake): each band holds a few modes; fs_fake ~ 2.5*f_hi
BANDS = [(2, 120, 300.0), (120, 320, 800.0), (320, 560, 1400.0)]
ORDMAX = 100
EPS, MIN_PTS = 0.02, 8   # DBSCAN: modal-distance radius and core-point threshold

MPE_CACHE_DIR = OUT_DIR / "mpe_cache"
REFIT_MPE = False   # set True to ignore existing MPE cache files

`full_band_mpe` (in `modekit.bandmpe`) runs the whole procedure on one set:

1. **Band-wise pLSCF.** One pLSCF per sub-band of `BANDS`, at that band's fictitious sampling rate. The stable poles of a band are clustered into modes (DBSCAN with `EPS` and `MIN_PTS`, see below), and the modes of all the bands are combined into one pole set.
2. **LSFD.** One residue fit over the full band at the combined poles (see the LSFD section below). The hammer is the only reference, so the band fits already give the participation factor of each pole; the LSFD keeps those fixed and solves for the mode shapes only (`refs="global"`). This is the second step of PolyMAX.

The band fits are cached as Equinox files under `outputs/garteur/mpe_cache` (through `modekit.serialisation`), so reruns load them from disk. The name of a cache file carries a hash of the data and of the settings above, so changing either one refits. Set `REFIT_MPE = True` to refit anyway.

### Front taps

We fit every front set in the next cell; `full_band_mpe` reports the modes it found in each. All the figures of every set are saved under `figures/garteur/ema/<set>/`, and only those of the set named by `tap_id` are shown; change `tap_id` to look at another set.

In [ ]:
# the procedure above on every front set: one pass, the hammer is the only reference
FR_TAPS = ["fr_hs1", "fr_hs2", "fr_hsr", "fr_ds"]
tap_id = "fr_hs1"   # change manually: the set whose figures are shown

fr_fits, fr_band_data = {}, {}
for t in FR_TAPS:
    fr_fits[t], bds = full_band_mpe(
        [(frf[t].freqs, frf[t].H)], resp_win[t].alpha, fs, BANDS, spectrum="frf_tap",
        quantity="acceleration", ordmax=ORDMAX, sc=SC, hc=HC, eps=EPS, min_pts=MIN_PTS,
        refs="global", cache=t, cache_dir=MPE_CACHE_DIR, refit=REFIT_MPE)
    fr_band_data[t] = bds[0]   # [BandFit, ...] of the one pass
    print(f"{t}: {fr_fits[t].model.Fn.size:2d} modes @ {jnp.round(fr_fits[t].model.Fn, 1)}")

# the modes each band of tap_id contributed
for flo, fhi, est, poles, res in fr_band_data[tap_id]:
    fn = res.Fn[(res.Fn >= flo) & (res.Fn <= fhi)]
    print(f"{tap_id} band {flo:4g}-{fhi:4g} Hz (fs_fake={est.fs:5g}): {len(fn)} modes @ {jnp.round(fn, 1)}")

Each sub-band is fitted on its own normalised axis, so it gets its own stabilisation chart: the model order on the right axis, the CMIF of the band's FRF overlaid on the left. Columns of stable poles mark the modes.

In [ ]:
# One stabilisation chart per sub-band, the CMIF of the band's FRF overlaid.
# est.freqs already holds the band's physical frequencies, so we slice the FRF to match
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_plot(poles, data=frf[t].H[:, :, sel], indicator="cmif", xlim=(flo, fhi),
                                  figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/stab_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

Within each band the stable poles are grouped into modes with DBSCAN on the modal distance (normalised frequency gap plus $1 - \mathrm{MAC}$), and the representative pole of each cluster is kept. The kept poles of every band are then combined into one pole set.

In [ ]:
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_cluster_plot(res, data=frf[t].H[:, :, sel], xlim=(flo, fhi), cmap_name="Dark2",
                                          figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/cluster_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

To pick `eps` (how close two poles must be, in modal distance, to count as the same mode) we use the k-distance graph: for every pole we measure its modal distance to its `min_pts`-th nearest neighbour and sort these distances from largest to smallest.

Poles of a real mode sit close together across model orders, so their distances are small: the flat, low plateau on the right. Noise poles are scattered, so their distances are large: the steep rise on the left. The knee between the two is a good value for `eps`.

In [ ]:
# k-distance curve per band: the knee is a good eps
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        fig, axes = est.k_distance_plot(poles, min_pts=MIN_PTS, eps=EPS, figsize=(6, 4),
                                        save_fig_name=FIG_DIR / f"ema/{t}/kdist_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# Frequency-damping cluster chart per band;
# a tight vertical group means consistent damping across model orders.
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        fig, axes = est.cluster_freqs_zetas_plot(res, xlim=(flo, fhi), figsize=(6, 4),
                                                 save_fig_name=FIG_DIR / f"ema/{t}/fzeta_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

#### LSFD

The combined poles fix the frequencies and the damping. LSFD holds them fixed and fits the modal constants (the residues $A$) plus the lower and upper residuals for the out-of-band modes, in one band over the whole identified range (2–560 Hz; see Avitabile Fig. 9.18). Fitting the residues jointly over the full band, rather than band by band, gives the tightest overall synthesis. `full_band_mpe` ran this fit above: each `fr_fits[t]` holds the LSFD estimator and the modal model it fitted.

In [ ]:
# the LSFD fit of tap_id; it holds the combined Fns and Zetas
lsfd, model = fr_fits[tap_id]
print("fn   [Hz]:", jnp.round(model.Fn, 2))
print("zeta [%] :", jnp.round(model.Zeta * 100, 3))

The synthesis compares the measured FRFs with the ones rebuilt from the identified modes, one panel per sensor; the unit-normalised real mode shapes follow, one column per mode.

In [ ]:
# Measured vs synthesised FRF, one panel per sensor
for t in FR_TAPS:
    lsfd, model = fr_fits[t]
    fig, axes = lsfd.synthesis_plot(
        model,
        frf[t].H,
        xlim=lsfd.band,
        y_labels=[f"{lab}\n" + r"$|H|$ [(m/s$^2$)/N]" for lab in SENSORS],
        # phase=True,
        figsize=(6, 8),
        save_fig_name=FIG_DIR / f"ema/{t}/synthesis.pdf",
    )
    if t == tap_id:
        plt.show()
    else:
        plt.close(fig)

# Unit-normalised real shapes of tap_id
model = fr_fits[tap_id].model
shapes = model.mode_shapes(real=True, normalize="l2")  # (Q, M)
print("unit-normalised real mode shapes (Q x M):", shapes.shape)

# values: one row per sensor, one column per identified mode (labelled by fn)
print(f"{'sensor / f [Hz]':>16}" + "".join(f"{f:>9.1f}" for f in model.Fn))
for s, row in zip(SENSORS, shapes):
    print(f"{s:>16}" + "".join(f"{v:>9.4f}" for v in row))

We gather the modes of every front set in one table with `reporting.mode_table` and write it to `outputs/garteur/` in CSV, Excel and HTML with `export.save_table`.

In [ ]:
# one row per identified mode
fr_tap_df = rp.mode_table(fr_fits)
display(fr_tap_df.head())

# archive under OUT_DIR in CSV, Excel and HTML
save_table(fr_tap_df, OUT_DIR / "fr_tap_modes", fmt=["csv", "excel", "html"])

### Right-wing taps

The same procedure on the right-wing sets, with the band above 320 Hz split in two and a higher maximum model order. The figures are saved under `figures/garteur/ema/<set>/` as well.

In [ ]:
RW_TAPS = ["rw_hs2", "rw_hsr", "rw_ds"]
tap_id = "rw_ds"   # change manually: the set whose figures are shown

# the last front band split in two, and a higher order
BANDS_RW_TAP = [
    (2, 120, 300.0),
    (120, 320, 800.0),
    (320, 470, 1200.0),
    (470, 560, 1400.0),
]
ORDMAX_RW_TAP = 150

rw_fits, rw_band_data = {}, {}
for t in RW_TAPS:
    rw_fits[t], bds = full_band_mpe(
        [(frf[t].freqs, frf[t].H)], resp_win[t].alpha, fs, BANDS_RW_TAP, spectrum="frf_tap",
        quantity="acceleration", ordmax=ORDMAX_RW_TAP, sc=SC, hc=HC, eps=EPS, min_pts=MIN_PTS,
        refs="global", cache=t, cache_dir=MPE_CACHE_DIR, refit=REFIT_MPE)
    rw_band_data[t] = bds[0]   # [BandFit, ...] of the one pass
    print(f"{t}: {rw_fits[t].model.Fn.size:2d} modes @ {jnp.round(rw_fits[t].model.Fn, 1)}")

# the modes each band of tap_id contributed
for flo, fhi, est, poles, res in rw_band_data[tap_id]:
    fn = res.Fn[(res.Fn >= flo) & (res.Fn <= fhi)]
    print(f"{tap_id} band {flo:4g}-{fhi:4g} Hz (fs_fake={est.fs:5g}): {len(fn)} modes @ {jnp.round(fn, 1)}")

In [ ]:
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_plot(poles, data=frf[t].H[:, :, sel], indicator="cmif", xlim=(flo, fhi),
                                  figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/stab_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_cluster_plot(res, data=frf[t].H[:, :, sel], xlim=(flo, fhi), cmap_name="Dark2",
                                          figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/cluster_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# k-distance curve per band: the knee is a good eps
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        fig, axes = est.k_distance_plot(poles, min_pts=MIN_PTS, eps=EPS, figsize=(6, 4),
                                        save_fig_name=FIG_DIR / f"ema/{t}/kdist_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# Frequency-damping cluster chart per band;
# a tight vertical group means consistent damping across model orders.
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        fig, axes = est.cluster_freqs_zetas_plot(res, xlim=(flo, fhi), figsize=(6, 4),
                                                 save_fig_name=FIG_DIR / f"ema/{t}/fzeta_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# the LSFD fit of tap_id; it holds the combined Fns and Zetas
lsfd, model = rw_fits[tap_id]
print("fn   [Hz]:", jnp.round(model.Fn, 2))
print("zeta [%] :", jnp.round(model.Zeta * 100, 3))

In [ ]:
for t in RW_TAPS:
    lsfd, model = rw_fits[t]
    fig, axes = lsfd.synthesis_plot(
        model,
        frf[t].H,
        xlim=lsfd.band,
        y_labels=[f"{lab}\n" + r"$|H|$ [(m/s$^2$)/N]" for lab in SENSORS],
        # phase=True,
        figsize=(6, 8),
        save_fig_name=FIG_DIR / f"ema/{t}/synthesis.pdf",
    )
    if t == tap_id:
        plt.show()
    else:
        plt.close(fig)

# Unit-normalised real shapes of tap_id
model = rw_fits[tap_id].model
shapes = model.mode_shapes(real=True, normalize="l2")  # (Q, M)
print("unit-normalised real mode shapes (Q x M):", shapes.shape)

# values: one row per sensor, one column per identified mode (labelled by fn)
print(f"{'sensor / f [Hz]':>16}" + "".join(f"{f:>9.1f}" for f in model.Fn))
for s, row in zip(SENSORS, shapes):
    print(f"{s:>16}" + "".join(f"{v:>9.4f}" for v in row))

In [ ]:
# one row per identified mode
rw_tap_df = rp.mode_table(rw_fits)
display(rw_tap_df.head())

# archive under OUT_DIR in CSV, Excel and HTML
save_table(rw_tap_df, OUT_DIR / "rw_tap_modes", fmt=["csv", "excel", "html"])

## Output-only analysis of the fan-excited records

The remaining sets were recorded with a fan blowing on the structure: `nb` with nothing in front of the fan, `b0` and `b45` with a bar in front of the structure at 0° and 45°, which disturbs the flow; `hs` and `ds` are again the healthy and the damaged state. In these sets the hammer only triggered the acquisition and plays no part in the analysis. We treat the responses as output-only data and run the same identification on their spectra, as one would with ambient vibration measurements. A fan is not the broadband excitation an operational modal analysis assumes, so this part demonstrates the output-only pipeline rather than a proper OMA.

In [ ]:
ftaps = {
    "nb_hs": data["fanfr_nb_hs"],
    "nb_ds": data["fanfr_nb_ds"],
    "b0_hs": data["fanfr_b0_hs"],
    "b0_ds": data["fanfr_b0_ds"],
    "b45_hs": data["fanfr_b45_hs"],
    "b45_ds": data["fanfr_b45_ds"],
}

# readable names of the fan sets, for the legends
FTAP_LABELS = {
    "nb_hs": "no bar, healthy", "nb_ds": "no bar, damaged",
    "b0_hs": "bar at 0°, healthy", "b0_ds": "bar at 0°, damaged",
    "b45_hs": "bar at 45°, healthy", "b45_ds": "bar at 45°, damaged",
}

We plot the responses of one set (`nb_hs`), one column per accelerometer. The hammer channel is in the files but is not used from here on.

In [ ]:
cmap = plt.get_cmap("Dark2")

figs = {}
# for name, tap in ftaps.items():
for name, ftap in {k: ftaps[k] for k in ["nb_hs"]}.items():
    n_reps = ftap.data.shape[0]
    n_cols = len(ftap.channels[1:])

    fig, axes = plt.subplots(
        n_reps, n_cols,
        figsize=(4 * n_cols, 2 * n_reps),
        sharex=True, squeeze=False,
    )
    for j, ch in enumerate(ftap.channels[1:]):
        sig = ftap.load_channel(ch)
        color = cmap(j)   # one color per sensor 
        for i in range(n_reps):
            axes[i, j].plot(time_array, sig[i, :], color=color, linewidth=1.0)
            axes[i, j].grid(True, linestyle="--", alpha=0.7)
        axes[0, j].set_title(ch)

    for i in range(n_reps):
        axes[i, 0].set_ylabel(f"rep {i + 1}")
    for j in range(n_cols):
        axes[-1, j].set_xlabel("Time [s]")

    fig.suptitle(FTAP_LABELS[name])
    fig.tight_layout()
    save_path = FIG_DIR / f"oma/responses_{name}.pdf"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    figs[name] = (fig, axes)

### Preprocessing

The records are sampled at 16384 Hz, far above the modal band. Two steps bring them down to the band of interest: detrending and decimation by 8, to 2048 Hz, which keeps the 2–560 Hz band of the EMA well below the new Nyquist frequency; and a zero-phase high-pass at 5 Hz to remove the low-frequency drift. Zero-phase filtering preserves the relative phase between the channels, on which the mode shapes depend.

In [ ]:
from modekit import preprocess

Q_DEC = 8  # 16384 Hz -> 2048 Hz; Nyquist 1024 Hz keeps the 2-560 Hz band (used in EMA)
ftaps_proc = {}
for name, ftap in ftaps.items():
    y = preprocess.detrend_data(ftap.data[:, 1:, :], type="constant")  # responses only
    y, fs_dec = preprocess.decimate_data(y, fs, Q_DEC)
    y = preprocess.filter_data(y, fs_dec, Wn=5.0, order=8, btype="highpass")
    ftaps_proc[name] = y

print(f"decimated: fs = {fs_dec:g} Hz, shape = {ftaps_proc['nb_hs'].shape}")

`OmaModel` holds the processed responses and computes their correlations and spectra. With `estimator="cor"` the spectra are correlograms: the correlation functions up to `n_lags` lags, tapered by an exponential window whose decay rate the model exposes as `window_rate`. With `n_lags = 4096` at 2048 Hz their frequency resolution is 0.25 Hz.

In [ ]:
from modekit.freq_models import OmaModel
import equinox as eqx

# n_lags << nt; half-spectra resolution fs_dec / (2 * n_lags) = 0.25 Hz
ftap_models = {
    name: OmaModel(ftaps_proc[name], fs_dec, fft_args={"n_lags": 4096}, estimator="cor")
    for name in ftaps_proc
}

print("\nOmaModel fields:")
print("-" * 30)
eqx.tree_pprint(ftap_models["nb_hs"])

### Autocorrelation of the responses

By the Wiener–Khinchin theorem the autocorrelation and the power spectrum are a Fourier pair, so under broadband excitation the output correlation functions look like free decays: the same information as an impulse response. That is the value of the correlogram: it lets the time-domain algorithms of classical modal analysis work on output-only data, once it is converted to correlation functions.

`get_ac` returns the correlations the `'cor'` half spectra are built from, before the exponential window. The rule for `n_lags`: long enough for the correlations to decay, but only a small fraction of the record length (10–20 % of `nt`), because the biased estimate averages fewer products as the lag grows and its variance blows up as the lag approaches `nt`.

In [ ]:
AC = namedtuple("AC", ["lags", "R"])
acs = {name: AC(*m.get_ac(n_lags=4096, normalize=True)) for name, m in ftap_models.items()}

fig, axes = plot_signal(
    np.asarray(acs["nb_hs"].lags),
    {FTAP_LABELS[name]: np.asarray(ac.R) for name, ac in acs.items()},
    x_label=r"Lag $\tau$ [s]",
    y_labels=[rf"$\rho_{{\ddot{{y}}_{{{q}}}}}(\tau)$ [-]" for q in range(1, np.shape(acs["nb_hs"].R)[0] + 1)],
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/autocorrelation.pdf",
)

### Output spectra after preprocessing

We plot the output PSDs through `get_psd` (the diagonal of $S_{yy} = S^+ + S^{+H}$), as for the `EmaModel`, over the full band and zoomed on the identified one.

In [ ]:
OPSD = namedtuple("OPSD", "freqs Syy")
opsd = {name: OPSD(*m.get_psd()) for name, m in ftap_models.items()}

# abs: the correlogram diagonal can dip below zero in the decimation
fig, axes = plot_signal(
    opsd["nb_hs"].freqs,
    {FTAP_LABELS[name]: np.abs(p.Syy) for name, p in opsd.items()},
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{q}}}}}$ [(m/s$^2$)$^2$]" for q in range(1, opsd["nb_hs"].Syy.shape[0] + 1)],
    yscale="log",
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/Syy.pdf",
)
plt.show()

# zoomed-view on the identified band
fig, axes = plot_signal(
    opsd["nb_hs"].freqs,
    {FTAP_LABELS[name]: np.abs(p.Syy) for name, p in opsd.items()},
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{q}}}}}$ [(m/s$^2$)$^2$]" for q in range(1, opsd["nb_hs"].Syy.shape[0] + 1)],
    yscale="log",
    xlim=(5, 400),
    ylim=(1e-5, 10),
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/Syy_zoom.pdf",
)
plt.show()

### Periodogram versus correlogram

The same processed records through both estimators. `'per'` is Welch averaging (Hann window, 4096-sample segments, 50 % overlap: $\Delta f = 0.5$ Hz with about 16 averages); `'cor'` is the exponential-windowed correlogram ($\Delta f = 0.25$ Hz, drawn on the shared 0.5 Hz grid and scaled by $2/f_s$ to density units). The window damping of the correlogram flattens and widens the peaks (the energy is conserved, so the peak drops and the baseline around it rises). pLSCF's `sd_cor` correction removes that trade, while Welch keeps the sharper peaks at the cost of a noisier baseline.

In [ ]:
psd_per = OPSD(*OmaModel(
    ftaps_proc["nb_hs"], fs_dec, fft_args={"nperseg": 4096}, estimator="per"
).get_psd())
psd_cor = opsd["nb_hs"]

fig, axes = plot_signal(
    psd_per.freqs,
    {
        "per (Welch)": np.asarray(psd_per.Syy),
        # correlogram scaled to density units [(m/s^2)^2/Hz]
        "cor (correlogram)": np.abs(psd_cor.Syy[:, ::2]) * 2 / fs_dec,
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{\ddot{{y}}_{{{q}}}}}$ [(m/s$^2$)$^2$/Hz]" for q in range(1, np.shape(psd_per.Syy)[0] + 1)],
    yscale="log",
    xlim=(5, 400),
    ylim=(1e-8, 1e-2),
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/Syy_per_vs_cor_nb_hs.pdf",
)
plt.show()

### pLSCF on the half spectra

We run the same band-wise procedure as for the EMA sets, with the half spectra in the role of the FRFs and `spectrum="sd_cor"`, so that pLSCF removes exactly the artificial damping of the exponential window (see `get_output_spectra`). The references are fitted in two separate passes, the front sensor (`[0]`) and the wing pair (`[2, 3]`); the tail stays an output row throughout, so its mode-shape component is kept. The two mode sets are merged by nearest frequency: a mode found by both passes is counted once and kept from the pass that identifies it more robustly, the one with the larger stabilisation cluster.

The residue (LSFD) fit then uses the merged poles on the spectra against the union of the reference columns. No band fit carries the factors of a pole on that union, because the passes used different references, so `full_band_mpe` re-estimates them with a local pLSCF model per merged pole on the union spectra (`refs="local"`), and the LSFD solves for the shapes only. The figures of every fan set are saved under `figures/garteur/oma/<set>/`; the set named by `ftap_id` is shown.

In [ ]:
OS = namedtuple("OS", "freqs S")

# two reference passes, fitted separately then merged; tail = output row only
REF_PASSES = {
    "front": [0],
    "wings": [2, 3],
}
REFS_LSFD = [c for refs in REF_PASSES.values() for c in refs]   # union -> LSFD residue fit

# per fan set: the half-spectra of each pass, and of the union for the LSFD
os_pass = {
    name: {p: OS(*m.get_output_spectra(ref_dofs=refs)) for p, refs in REF_PASSES.items()}
    for name, m in ftap_models.items()
}
os_lsfd = {name: OS(*m.get_output_spectra(ref_dofs=REFS_LSFD)) for name, m in ftap_models.items()}

W_RATE = ftap_models["nb_hs"].window_rate  # window damping [1/s]
BANDS_FTAP = [
    (5, 120, 300.0),
    (120, 320, 800.0),
    (320, 400, 1000.0),
]
ORDMAX_FTAP = 150
# the local pLSCF models that give the factors; at order 2 they miss weak modes
# that only the wing references find, so order 4
LOCAL_KW = dict(order=4, deltaf=2.0, rtol=0.03)

In [ ]:
FTAPS = ["nb_hs", "nb_ds"]
ftap_id = "nb_ds"   # change manually: the set whose figures are shown

# the procedure on every fitted fan set: one band-wise pLSCF per reference pass,
# the passes' poles merged, then the LSFD on the union spectra
ftap_fits, ftap_band_data = {}, {}
for t in FTAPS:
    ftap_fits[t], bds = full_band_mpe(
        [(os_pass[t][p].freqs, os_pass[t][p].S) for p in REF_PASSES], W_RATE, fs_dec, BANDS_FTAP,
        spectrum="sd_cor", lsfd_data=(os_lsfd[t].freqs, os_lsfd[t].S),
        ordmax=ORDMAX_FTAP, sc=SC, hc=HC, eps=EPS, min_pts=MIN_PTS,
        refs="local", local_kw=LOCAL_KW,
        cache=[f"{t}_{p}" for p in REF_PASSES], cache_dir=MPE_CACHE_DIR, refit=REFIT_MPE)
    ftap_band_data[t] = list(zip(REF_PASSES, bds))   # [(pass, [BandFit, ...]), ...]
    fn = np.asarray(ftap_fits[t].model.Fn)
    print(f"{t}: {int(np.isfinite(fn).sum()):2d} modes @ {np.round(fn, 1)}")

# the modes each pass of ftap_id found, before the merge
for p, bd in ftap_band_data[ftap_id]:
    fn = np.concatenate([np.asarray(res.Fn)[np.asarray((res.Fn >= flo) & (res.Fn <= fhi))]
                         for flo, fhi, est, poles, res in bd])
    print(f"{ftap_id} {p:>5} refs: {fn.size:2d} modes @ {np.round(fn, 1)}")

In [ ]:
for t in FTAPS:
    for p, bd in ftap_band_data[t]:
        sp = os_pass[t][p]
        for flo, fhi, est, poles, res in bd:
            sel = (sp.freqs >= flo) & (sp.freqs <= fhi)
            fig, axes = est.stab_plot(
                poles, data=np.asarray(sp.S)[:, :, sel], indicator="cmif",
                xlim=(flo, fhi), figsize=(6, 4),
                save_fig_name=FIG_DIR / f"oma/{t}/stab_{p}_{flo:g}-{fhi:g}.pdf")
            fig.suptitle(f"{p} refs", y=1.02)
            if t == ftap_id:
                plt.show()
            else:
                plt.close(fig)

In [ ]:
for t in FTAPS:
    for p, bd in ftap_band_data[t]:
        sp = os_pass[t][p]
        for flo, fhi, est, poles, res in bd:
            sel = (sp.freqs >= flo) & (sp.freqs <= fhi)
            fig, axes = est.stab_cluster_plot(
                res, data=np.asarray(sp.S)[:, :, sel], xlim=(flo, fhi),
                cmap_name="Dark2", figsize=(6, 4),
                save_fig_name=FIG_DIR / f"oma/{t}/cluster_{p}_{flo:g}-{fhi:g}.pdf")
            fig.suptitle(f"{p} refs", y=1.02)
            if t == ftap_id:
                plt.show()
            else:
                plt.close(fig)

In [ ]:
for t in FTAPS:
    for p, bd in ftap_band_data[t]:
        for flo, fhi, est, poles, res in bd:
            fig, axes = est.cluster_freqs_zetas_plot(
                res, xlim=(flo, fhi), figsize=(6, 4),
                save_fig_name=FIG_DIR / f"oma/{t}/fzeta_{p}_{flo:g}-{fhi:g}.pdf")
            fig.suptitle(f"{p} refs", y=1.02)
            if t == ftap_id:
                plt.show()
            else:
                plt.close(fig)

In [ ]:
# the LSFD fit of ftap_id: the merged poles over the full band, on the spectra
# against the union of the reference columns
lsfd, model = ftap_fits[ftap_id]
print("fn   [Hz]:", jnp.round(model.Fn, 2))
print("zeta [%] :", jnp.round(model.Zeta * 100, 3))

for t in FTAPS:
    lsfd, model = ftap_fits[t]
    fig, axes = lsfd.synthesis_plot(
        model,
        os_lsfd[t].S,
        pairs=[(q, 0) for q in range(len(SENSORS))],  # every output against the front reference
        xlim=lsfd.band,
        y_labels=[f"{lab}\n" + r"$|S^+|$" for lab in SENSORS],
        figsize=(6, 8),
        save_fig_name=FIG_DIR / f"oma/{t}/synthesis.pdf",
    )
    if t == ftap_id:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# one row per identified mode, across the fitted fan sets
ftap_df = rp.mode_table(ftap_fits)
display(ftap_df.head(12))

save_table(ftap_df, OUT_DIR / "fan_tap_modes", fmt=["csv", "excel", "html"])